# 06 — Panel Regression: What Drives the Renewable Transition?

Notebook 04 showed time alone can predict the EU-wide renewable share with reasonable accuracy.
Notebook 05 added two economic drivers — GDP per capita and household electricity prices — across all 27 member states.

Here those threads converge: a regularised panel regression that answers *why* some countries move faster than others,
evaluated with expanding-window cross-validation so test data is always in the future relative to the training window.

In [1]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.base import clone
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.config import DB_PATH
from utils.charts import COLORS

## 1. Load the enriched panel

In [2]:
conn = sqlite3.connect(DB_PATH)
panel = pd.read_sql(
    "SELECT * FROM country_panel ORDER BY country, year",
    conn,
)
conn.close()

print(f"{panel['country'].nunique()} countries x {panel['year'].nunique()} years = {len(panel)} rows")
print(f"Years: {panel['year'].min()}–{panel['year'].max()}")
panel.head()

27 countries x 18 years = 485 rows
Years: 2007–2024


,year,country,renewable_share,dependency_rate,gdp_pps,elec_price_eur_kwh,t
0,2007,Austria,96.72,68.48,30945.9,0.17400,2
1,2008,Austria,97.36,68.74,31853.9,0.17755,3
2,2009,Austria,95.35,65.12,30722.4,0.19090,4
3,2010,Austria,95.44,62.78,31605.6,0.19485,5
4,2011,Austria,94.37,69.98,32928.8,0.19755,6


## 2. Build the feature matrix

The model has three continuous predictors plus a fixed effect for each country.
Country fixed effects are encoded as 26 binary dummies (Austria absorbed into the intercept).
One dummy per country captures structural differences in renewable mix that are not explained by GDP or price —
geography, policy history, grid infrastructure.

In [3]:
country_dummies = pd.get_dummies(panel["country"], drop_first=True, prefix="c", dtype=float)
ref_country = sorted(panel["country"].unique())[0]

continuous = ["t", "gdp_pps", "elec_price_eur_kwh"]
X = pd.concat([panel[continuous], country_dummies], axis=1)
y = panel["renewable_share"].values
years_arr = panel["year"].values

print(f"Feature matrix: {X.shape[0]} rows x {X.shape[1]} features")
print(f"  Reference country (intercept): {ref_country}")
print(f"  Continuous: {continuous}")
print(f"  Country dummies: {country_dummies.shape[1]}")

Feature matrix: 485 rows x 29 features
  Reference country (intercept): Austria
  Continuous: ['t', 'gdp_pps', 'elec_price_eur_kwh']
  Country dummies: 26


## 3. Expanding-window cross-validation

Standard k-fold would leak future data into training. Instead, each fold trains on all years up to
year *t* and tests on year *t+1*. The window starts after 5 training years (2007–2011 → test 2012)
and expands one year at a time through 2024.

Three models compete:
- **Naive (persistence):** each country’s value from the prior year
- **Ridge (α=1.0):** L2 regularisation, all features active
- **Lasso (α=0.1):** L1 regularisation, sparse solution

In [4]:
MIN_TRAIN = 5
unique_years = sorted(set(years_arr))
X_arr = X.values


def run_model_cv(model, X_arr, y_arr, years_arr, unique_years, min_train=MIN_TRAIN):
    records = []
    for i in range(min_train, len(unique_years)):
        test_year = unique_years[i]
        tr = years_arr < test_year
        te = years_arr == test_year
        m = clone(model)
        m.fit(X_arr[tr], y_arr[tr])
        pred = m.predict(X_arr[te])
        records.append({
            "year": test_year,
            "MAE": mean_absolute_error(y_arr[te], pred),
            "RMSE": mean_squared_error(y_arr[te], pred) ** 0.5,
        })
    return pd.DataFrame(records)


def run_naive_cv(panel_df, target, unique_years, min_train=MIN_TRAIN):
    records = []
    for i in range(min_train, len(unique_years)):
        test_year = unique_years[i]
        prev_year = unique_years[i - 1]
        prev = panel_df[panel_df["year"] == prev_year].set_index("country")[target]
        test = panel_df[panel_df["year"] == test_year].set_index("country")[target]
        common = prev.index.intersection(test.index)
        records.append({
            "year": test_year,
            "MAE": mean_absolute_error(test[common], prev[common]),
            "RMSE": mean_squared_error(test[common], prev[common]) ** 0.5,
        })
    return pd.DataFrame(records)


ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10_000))

cv_naive = run_naive_cv(panel, "renewable_share", unique_years)
cv_ridge = run_model_cv(ridge, X_arr, y, years_arr, unique_years)
cv_lasso = run_model_cv(lasso, X_arr, y, years_arr, unique_years)

print("CV complete.")

CV complete.


In [5]:
summary = pd.DataFrame({
    "Naive (persistence)": cv_naive[["MAE", "RMSE"]].mean(),
    "Ridge (\u03b1=1.0)": cv_ridge[["MAE", "RMSE"]].mean(),
    "Lasso (\u03b1=0.1)": cv_lasso[["MAE", "RMSE"]].mean(),
}).T.round(3)

summary.index.name = "Model"
print("Mean CV error across all test folds (percentage points):")
display(summary)

Mean CV error across all test folds (percentage points):


,MAE,RMSE
Model,,
Naive (persistence),2.206,3.609
Ridge (α=1.0),4.312,6.377
Lasso (α=0.1),4.296,6.494


**Note on the baseline:** Renewable share is a slow-moving series — most countries shift by only 1–2 pp per year.
A persistence forecast (predict tomorrow = today) is therefore very hard to beat on annual MAE.
The panel regression captures the *structural* relationship between economic conditions and renewable mix,
which is where its value lies: explaining *why* countries differ, not predicting next year's increment.

In [6]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cv_naive["year"], y=cv_naive["MAE"],
    name="Naive (persistence)",
    mode="lines+markers",
    line=dict(color=COLORS["fossil"], width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=cv_ridge["year"], y=cv_ridge["MAE"],
    name="Ridge",
    mode="lines+markers",
    line=dict(color=COLORS["renewables"], width=2),
))
fig.add_trace(go.Scatter(
    x=cv_lasso["year"], y=cv_lasso["MAE"],
    name="Lasso",
    mode="lines+markers",
    line=dict(color=COLORS["neutral"], width=2),
))
fig.update_layout(
    title="Expanding-window CV — MAE by test year",
    yaxis_title="MAE (percentage points)",
    xaxis_title=None,
    plot_bgcolor="white",
    yaxis=dict(gridcolor="#eeeeee"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.show()

## 4. What drives the renewable transition?

Ridge is refit on the full dataset to extract coefficients.
All features are standardised before fitting, so coefficients are comparable:
a coefficient of 5 means a one-standard-deviation increase in that feature
is associated with a 5 percentage-point change in renewable share,
holding everything else constant.

In [7]:
ridge_full = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge_full.fit(X_arr, y)

coef = ridge_full.named_steps["ridge"].coef_
feat_names = X.columns.tolist()

importance = pd.DataFrame({"feature": feat_names, "coef": coef})
importance["abs_coef"] = importance["coef"].abs()

econ = importance[~importance["feature"].str.startswith("c_")].copy()
dummies = importance[importance["feature"].str.startswith("c_")].sort_values("coef")

print("Economic feature coefficients (standardised Ridge):")
display(econ[["feature", "coef"]].round(3).set_index("feature"))

print(f"\nCountry fixed effects (top 5 positive / bottom 5 negative vs Austria):")
display(
    pd.concat([dummies.tail(5), dummies.head(5)])[["feature", "coef"]]
    .assign(feature=lambda d: d["feature"].str.replace("c_", "", regex=False))
    .set_index("feature")
    .round(3)
)

Economic feature coefficients (standardised Ridge):


,coef
feature,
t,0.247
gdp_pps,-1.354
elec_price_eur_kwh,1.359



Country fixed effects (top 5 positive / bottom 5 negative vs Austria):


,coef
feature,
Lithuania,1.301
Croatia,2.094
Netherlands,2.211
Estonia,7.056
Latvia,8.691
Luxembourg,-5.489
Denmark,-4.829
Cyprus,-3.318
Malta,-3.151


In [8]:
# Show economic features + top/bottom 8 country dummies
top_dummies = pd.concat([dummies.head(8), dummies.tail(8)])
show = pd.concat([econ, top_dummies]).sort_values("coef").reset_index(drop=True)
labels = show["feature"].apply(lambda x: x.removeprefix("c_"))
bar_colors = [COLORS["renewables"] if c >= 0 else COLORS["fossil"] for c in show["coef"]]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=show["coef"],
    y=labels,
    orientation="h",
    marker_color=bar_colors,
))
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(
    title="Ridge standardised coefficients — economic drivers + top/bottom 8 country effects",
    xaxis_title="Standardised coefficient (pp change per 1-SD increase)",
    yaxis_title=None,
    plot_bgcolor="white",
    xaxis=dict(gridcolor="#eeeeee"),
    height=600,
    showlegend=False,
)
fig.show()

In [9]:
# Lasso feature selection: which features does L1 zero out?
lasso_full = make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10_000))
lasso_full.fit(X_arr, y)

lasso_coef = lasso_full.named_steps["lasso"].coef_
kept = [(feat_names[i].removeprefix("c_"), round(lasso_coef[i], 3))
        for i in range(len(feat_names)) if abs(lasso_coef[i]) > 1e-4]
zeroed = [feat_names[i].removeprefix("c_")
          for i in range(len(feat_names)) if abs(lasso_coef[i]) <= 1e-4]

print(f"Lasso keeps {len(kept)} features, zeroes {len(zeroed)}:")
print("\nKept (feature, coef):")
for name, val in sorted(kept, key=lambda x: -abs(x[1])):
    print(f"  {name:<30} {val:+.3f}")

if zeroed:
    print(f"\nZeroed out: {zeroed}")

Lasso keeps 28 features, zeroes 1:

Kept (feature, coef):
  Latvia                         +8.499
  Estonia                        +6.817
  Luxembourg                     -5.492
  Denmark                        -4.508
  Malta                          -3.220
  Cyprus                         -3.188
  Belgium                        -2.893
  Netherlands                    +2.055
  Croatia                        +1.826
  gdp_pps                        -1.315
  Ireland                        -1.286
  Italy                          -1.273
  Lithuania                      +1.049
  Hungary                        +1.025
  Portugal                       +0.714
  Slovakia                       +0.684
  Czechia                        +0.645
  Bulgaria                       +0.641
  elec_price_eur_kwh             +0.579
  Greece                         -0.474
  t                              +0.450
  Slovenia                       -0.441
  Sweden                         -0.209
  Finland             

## 5. Where does the model struggle?

Mean residual per country on the full training set:
positive means the model consistently under-predicts that country’s renewable share
(geography or policy captured poorly by the three continuous features),
negative means it over-predicts.

In [10]:
panel_res = panel.copy()
panel_res["predicted"] = ridge_full.predict(X_arr)
panel_res["residual"] = panel_res["renewable_share"] - panel_res["predicted"]

mean_res = panel_res.groupby("country")["residual"].mean().sort_values()
bar_colors2 = [COLORS["renewables"] if r >= 0 else COLORS["fossil"] for r in mean_res]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=mean_res.values,
    y=mean_res.index,
    orientation="h",
    marker_color=bar_colors2,
    text=mean_res.values.round(1),
    textposition="outside",
))
fig.add_vline(x=0, line_color="black", line_width=1)
fig.update_layout(
    title="Mean residual by country — Ridge fit on full data",
    xaxis_title="Mean residual (pp, positive = under-predicted)",
    yaxis_title=None,
    plot_bgcolor="white",
    xaxis=dict(gridcolor="#eeeeee"),
    height=700,
    showlegend=False,
    margin=dict(l=150, r=60),
)
fig.show()